# Display checks

This notebook checks the string renderers and the public `.show()` display methods. Renderer functions return text for reuse in reports; object methods print that text directly for convenient notebook inspection.

In [1]:
# reload notebook imports upon module updates
%load_ext autoreload
%autoreload 2

In [2]:
from contextlib import redirect_stdout
from io import StringIO
from pathlib import Path
import importlib
import sys

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / 'src').is_dir():
    project_root = project_root.parent
if not (project_root / 'src' / 'ensemblelab').is_dir():
    raise RuntimeError('Open this notebook from within the ensemblelab repository.')
sys.path.insert(0, str(project_root / 'src'))

import ensemblelab.generators as generators
import ensemblelab.display.summaries as summaries
import ensemblelab.optimizers.base as optimizer_base
import ensemblelab.optimizers.mmff as mmff
importlib.reload(summaries)
importlib.reload(generators)
importlib.reload(optimizer_base)
importlib.reload(mmff)

from ensemblelab.generators import Ensemble, generate
from ensemblelab.optimizers.mmff import MMFFOptimizer
from ensemblelab.display.summaries import (
    conformer_summary,
    ensemble_history,
    ensemble_summary,
)

## Default ensemble display

An unoptimized ensemble reports the molecule summary and conformer table. Energy and convergence fields are unavailable until optimization.

In [3]:
ensemble = generate('CCO', n_confs=2)
rendered = ensemble_summary(ensemble)
output = StringIO()
with redirect_stdout(output):
    show_result = ensemble.show()
result = output.getvalue()

assert isinstance(rendered, str)
assert show_result is None
assert result.strip() == rendered.strip()
assert 'Ensemble' in result
assert 'Energy: uncomputed' in result
assert 'Conformers' in result
assert 'N/A' in result

print(result, end='')

Ensemble
----------------------------
SMILES: CCO
Atoms: 9
Conformers: 2
Energy: uncomputed
Optimization: not run

Conformers
------------------------------------
+----+--------+--------------------+--------+-----------+-------+
| ID | Energy | Delta E (kcal/mol) | Method | Converged | Atoms |
+----+--------+--------------------+--------+-----------+-------+
| 0  | N/A    | N/A                | N/A    | N/A       | 9     |
| 1  | N/A    | N/A                | N/A    | N/A       | 9     |
+----+--------+--------------------+--------+-----------+-------+


## History and metadata sections

History and raw metadata can be requested together. The conformer table can be omitted when the additional sections are the focus. History is read from the canonical `ensemble.metadata['history']` workflow log.

In [4]:
output = StringIO()
with redirect_stdout(output):
    result = ensemble.show(history=True, metadata=True, conformers=False)
result = output.getvalue()

assert result.startswith('Ensemble\n')
assert 'Workflow History' in result
assert '1. generation' in result
assert 'Metadata' in result
assert 'Conformers\n' not in result
assert result.count('Ensemble') == 1

print(result, end='')

Ensemble
----------------------------
SMILES: CCO
Atoms: 9
Conformers: 2
Energy: uncomputed
Optimization: not run

Workflow History

1. generation
   Method: rdkit.ETKDGv3
   Requested: 2
   Generated: 2
   Random seed: 42

Metadata
+---------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------+
| Key                 | Value                                                                                                                                                       |
+---------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------+
| n_conformers        | 2                                                                                                                                                           |
| optimization_status | unoptimized    

## Conformer display

A conformer display reports fields held directly by the conformer object. This case uses water with one generated conformer.

In [61]:
conformer = generate('O', n_confs=1).conformers[0]
rendered = conformer_summary(conformer)
output = StringIO()
with redirect_stdout(output):
    result = conformer.show()
result = output.getvalue()

assert isinstance(rendered, str)
assert result == rendered + '\n'
assert 'Conformer 0' in result
assert 'Energy          N/A' in result
assert 'Atoms           3' in result

print(result, end='')

Conformer 0
Energy          N/A
Optimization    N/A
Converged       N/A
Atoms           3


## Optimized ensemble display

MMFF assigns energies, methods, and convergence states. The ensemble table reports relative energy from the lowest stored energy, and the canonical history records the generation and optimization events. `.show(history=True)` prints the combined multiline view.

In [6]:
optimized = MMFFOptimizer().optimize(generate('CCO', n_confs=2))
rendered = ensemble_summary(optimized)
output = StringIO()
with redirect_stdout(output):
    result = optimized.show(history=True)
result = output.getvalue()

assert isinstance(rendered, str)
assert result.startswith(rendered + '\n\n')
assert 'Delta E (kcal/mol)' in result
assert 'Optimization: MMFF' in result
assert '1. generation' in result
assert '2. optimization' in result

print(result, end='')

Ensemble
----------------------------
SMILES: CCO
Atoms: 9
Conformers: 2
Energy: computed
Optimization: MMFF

Conformers
------------------------------------
+----+-----------------+--------------------+--------+-----------+-------+
| ID | Energy          | Delta E (kcal/mol) | Method | Converged | Atoms |
+----+-----------------+--------------------+--------+-----------+-------+
| 0  | -1.337 kcal/mol | 0.000              | MMFF   | ✓         | 9     |
| 1  | -1.337 kcal/mol | 0.000              | MMFF   | ✓         | 9     |
+----+-----------------+--------------------+--------+-----------+-------+

Workflow History

1. generation
   Method: rdkit.ETKDGv3
   Requested: 2
   Generated: 2
   Random seed: 42

2. optimization
   Method: MMFF
   Input: 2
   Output: 2
   Converged: 2
   Unconverged: 0
   Max steps: 500
   Energy unit: kcal/mol


In [7]:
from ensemblelab.display.visualize import view

view(optimized.conformers[0])

## Canonical history display

The display reads optimization records from the canonical `metadata['history']` list. Generation, optimization, and filter records may contain different fields; only populated fields are rendered.

In [8]:
generated = generate('O', n_confs=1)
history_ensemble = Ensemble(
    smiles=generated.smiles,
    molecule=generated.molecule,
    conformers=generated.conformers,
    metadata={
        'history': [{'process': 'optimization', 'method': 'MMFF', 'max_steps': 500}],
    },
)
output = StringIO()
with redirect_stdout(output):
    result = history_ensemble.show(history=True, conformers=False)
result = output.getvalue()

assert '1. optimization' in result
assert 'Method: MMFF' in result
assert 'Max steps: 500' in result

print(result, end='')

Ensemble
----------------------------
SMILES: O
Atoms: 3
Conformers: 1
Energy: uncomputed
Optimization: not run

Workflow History

1. optimization
   Method: MMFF
   Max steps: 500
